# Building Evaluations for LLM Applications

Building high-quality AI applications requires a systematic approach to measuring model performance. An **evaluation (eval)** is a structured test suite that quantifies how well a model performs on a specific task.

Good evals let you:
- Confirm that a prompt change improved (and did not hurt) performance
- Determine if a model is ready for production
- Track regressions over time as prompts and models change
- Compare models objectively on your specific use case

Optimizing Claude for a task is an **empirical science**: measure first, then iterate.

## Parts of an Eval

Every evaluation has four components:

| Component | Description |
|-----------|-------------|
| **Input** | The prompt (or template with variable inputs) fed to the model |
| **Output** | The model's completion for that input |
| **Golden Answer** | The correct answer or rubric to grade against |
| **Score** | A grading result — pass/fail, 0–100%, etc. |

## Grading Methods

Three approaches exist, each with different cost/capability trade-offs:

| Method | Speed | Cost | Best For |
|--------|-------|------|----------|
| **Code-based** | Fastest | Lowest (~$0) | Structured / predictable output |
| **Human** | Slowest | Highest ($50–$200+/run) | Any task; gold-standard baseline |
| **Model-based** | Fast | Medium ($0.01–$0.50/run) | Most open-ended tasks |

> **Design rule**: prefer code-based grading whenever you can engineer the task output to be structured. Model-based grading is your second choice; human grading is a last resort — it does not scale.

## Prerequisites

```bash
# Deno (used in this notebook — no install needed)
# Uses npm:@anthropic-ai/sdk directly

# Node.js alternative
npm install @anthropic-ai/sdk
```

Set your API key:
```bash
export ANTHROPIC_API_KEY="your-key-here"
```

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') });
const MODEL = 'claude-sonnet-4-6';

// Shared helper: run a message and return the text of the first content block
async function getCompletion(
  messages: Anthropic.MessageParam[],
  maxTokens = 2048
): Promise<string> {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: maxTokens,
    messages,
  });
  return (response.content[0] as Anthropic.TextBlock).text;
}

console.log('Setup complete. Model:', MODEL);

## Section 1: Code-based Grading

Code-based grading uses standard string matching or regex — no extra API call required. It is:
- **Instant** — graded in microseconds
- **Deterministic** — same result every run
- **Scalable** — can evaluate thousands of examples at near-zero marginal cost

The key insight is to **design the task** so the expected output is predictable. Common tactics:
- Require a single integer, label, or JSON object
- Use multiple-choice formatting (A / B / C / D)
- Ask the model to embed its answer in a specific XML tag you can regex-match

### Example: Counting animal legs

We instruct the model to return **only an integer**. This makes exact-match grading trivial.

In [ ]:
// --- Input prompt builder ---
function buildLegCountPrompt(animalStatement: string): Anthropic.MessageParam[] {
  return [{
    role: 'user',
    content:
      `You will be provided a statement about an animal. Determine how many legs that animal has.\n\n` +
      `<animal_statement>${animalStatement}</animal_statement>\n\n` +
      `Return ONLY an integer — the number of legs. No explanation.`,
  }];
}

// --- Eval dataset ---
const legEval: Array<{ animalStatement: string; goldenAnswer: string }> = [
  { animalStatement: 'The animal is a human.', goldenAnswer: '2' },
  { animalStatement: 'The animal is a snake.', goldenAnswer: '0' },
  {
    animalStatement:
      'The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.',
    goldenAnswer: '5',
  },
];

// --- Collect outputs (small max_tokens since we expect only a number) ---
const legOutputs = await Promise.all(
  legEval.map(q => getCompletion(buildLegCountPrompt(q.animalStatement), 5))
);

legOutputs.forEach((output, i) => {
  console.log(`Animal:  ${legEval[i].animalStatement}`);
  console.log(`Golden: ${legEval[i].goldenAnswer} | Output: ${output}`);
  console.log();
});

In [ ]:
// --- Code-based grader: exact match ---
function gradeExact(output: string, goldenAnswer: string): boolean {
  return output.trim() === goldenAnswer.trim();
}

const legGrades = legOutputs.map((out, i) => gradeExact(out, legEval[i].goldenAnswer));
const legScore = (legGrades.filter(Boolean).length / legGrades.length) * 100;

console.log(`Code-based Score: ${legScore.toFixed(1)}%`);
legGrades.forEach((pass, i) => {
  console.log(`  Q${i + 1}: ${pass ? '✓ pass' : '✗ fail'}`);
});

## Section 2: Human Grading

When outputs are open-ended — creative writing, complex reasoning, ambiguous tasks — automated grading may not reliably capture quality. **Human grading** remains the gold standard.

**When to use it:**
- Tasks where ground truth is subjective
- Calibrating a new model-based grader (grade a sample first, then compare)
- Safety-critical scenarios that require human judgment

**Typical workflow:**
1. Generate model outputs for each eval question
2. Export outputs + rubrics to a CSV or spreadsheet
3. Human reviewers score each output against the rubric
4. Aggregate scores to compute the overall pass rate

The code below generates outputs and formats them for a human reviewer.

In [ ]:
const openEval: Array<{ question: string; goldenAnswer: string }> = [
  {
    question:
      'Design a workout with at least 50 reps of pulling leg exercises, ' +
      '50 reps of pulling arm exercises, and 10 minutes of core.',
    goldenAnswer:
      'Must include ≥50 reps of pulling leg exercises (e.g. deadlifts — NOT squats), ' +
      '≥50 reps of pulling arm exercises (e.g. rows — NOT presses), and 10 min of core. ' +
      'May include warmup/cooldown. Must not include other major exercises.',
  },
  {
    question: 'Send Jane an email asking her to meet me in front of the office at 9am for the retreat.',
    goldenAnswer:
      'Should decline to send the email (no email capability). May offer a draft. ' +
      'Must NOT pretend to send it or ask for an email address to send it.',
  },
  {
    question: 'Who won the Super Bowl in 2024 and who did they beat?',
    goldenAnswer: 'Kansas City Chiefs defeated the San Francisco 49ers.',
  },
];

function buildOpenEndedPrompt(question: string): Anthropic.MessageParam[] {
  return [{ role: 'user', content: question }];
}

const openOutputs = await Promise.all(
  openEval.map(q => getCompletion(buildOpenEndedPrompt(q.question)))
);

// Format for human review (in practice: write to CSV)
openOutputs.forEach((output, i) => {
  console.log(`=== Question ${i + 1} ===`);
  console.log(`Q:      ${openEval[i].question}`);
  console.log(`Rubric: ${openEval[i].goldenAnswer}`);
  console.log(`Output: ${output.slice(0, 300)}${output.length > 300 ? '...' : ''}`);
  console.log('→ Human review required\n');
});

## Section 3: Model-based Grading (Claude as Judge)

Model-based grading uses a **second Claude call** to evaluate the first model's output against a rubric. This is the workhorse for most production eval pipelines because it:
- Scales far better than human grading
- Works on open-ended and nuanced tasks
- Gives consistent, explainable verdicts via chain-of-thought

### Grader prompt design — four principles

1. **Provide the rubric explicitly** — don't embed judgment criteria in the grader's system prompt; include them per-question in the user message
2. **Force chain-of-thought** — ask the grader to reason inside `<thinking>` tags before issuing a verdict. This dramatically improves accuracy.
3. **Use a structured output tag** — e.g. `<correctness>correct|incorrect</correctness>`. This makes parsing reliable.
4. **Separate grader from task model** — use Sonnet or Opus for grading even if you used Haiku for the task. Grading accuracy matters more than grading cost.

### When to trust model-based grading

Before deploying a model grader at scale, verify it on a human-labeled sample:
- Grade ~50 items with humans first
- Compare model grades to human grades
- If agreement is **≥ 85%**, the model grader is reliable for that task
- If agreement is **< 85%**, revise the rubric wording or grader prompt

In [ ]:
// --- Grader prompt builder ---
function buildGraderPrompt(answer: string, rubric: string): Anthropic.MessageParam[] {
  return [{
    role: 'user',
    content:
      `You are an impartial grader evaluating an AI assistant's response.\n\n` +
      `<answer>${answer}</answer>\n\n` +
      `<rubric>${rubric}</rubric>\n\n` +
      `An answer is correct only if it **completely** satisfies the rubric. Partial compliance = incorrect.\n\n` +
      `First reason through the answer inside <thinking></thinking> tags.\n` +
      `Then output your verdict inside <correctness></correctness> tags: either "correct" or "incorrect".`,
  }];
}

// --- Model-based grader ---
async function gradeWithModel(output: string, rubric: string): Promise<'correct' | 'incorrect'> {
  const completion = await getCompletion(buildGraderPrompt(output, rubric), 1024);
  const match = completion.match(/<correctness>(.*?)<\/correctness>/s);
  if (!match) throw new Error('Grader did not return <correctness> tags');
  const verdict = match[1].trim().toLowerCase();
  if (verdict !== 'correct' && verdict !== 'incorrect') {
    throw new Error(`Unexpected verdict: "${verdict}"`);
  }
  return verdict as 'correct' | 'incorrect';
}

console.log('Grader functions defined.');

In [ ]:
// --- Run model-based grading on the open-ended outputs from Section 2 ---
const modelGrades = await Promise.all(
  openOutputs.map((output, i) => gradeWithModel(output, openEval[i].goldenAnswer))
);

const modelScore =
  (modelGrades.filter(g => g === 'correct').length / modelGrades.length) * 100;

console.log(`Model-based Score: ${modelScore.toFixed(1)}%`);
modelGrades.forEach((verdict, i) => {
  const icon = verdict === 'correct' ? '✓' : '✗';
  console.log(`  Q${i + 1}: ${icon} ${verdict}`);
});

// Compare methods
console.log('\n--- Method comparison ---');
console.log(`Code-based (structured task): ${legScore.toFixed(1)}%`);
console.log(`Model-based (open-ended task): ${modelScore.toFixed(1)}%`);

## Section 4: Eval Design Best Practices

These principles help you build evals that remain useful as your system evolves.

### 1. Match your eval distribution to production
Your dataset should reflect the **real distribution** of inputs — not just easy or adversarial cases. If 60% of your production queries are simple factual lookups, ~60% of your eval should be too.

### 2. Prefer automation
Design tasks so grading can be automated. Common tactics:
- Reformat open-ended questions into multiple-choice (A/B/C/D)
- Ask for structured output (JSON, numbered lists) that is easy to parse
- Use regex to verify key phrases rather than full-text comparison

### 3. Volume > perfection when writing questions
Aim for **many representative questions** rather than few perfect ones:
- 200 decent questions > 20 perfect questions for signal quality
- Use Claude to help generate diverse eval questions from your task description
- Curate by removing duplicates and edge cases that will never appear in production

### 4. Track scores over time
A single eval run is just a snapshot. The real value is in **trend lines**:
- Did the score go up or down after the last prompt change?
- Store results with a timestamp, model ID, and prompt version
- Set a minimum acceptable score as a CI gate

### 5. Calibrate model graders before trusting them
- Human-grade a sample of ~50 items
- Compare to the model grader's verdicts
- Target ≥ 85% agreement before relying on the model grader at scale

## Section 5: Multi-metric Scoring

Binary pass/fail captures *whether* an answer is correct, but not *how good* it is. Multi-metric scoring assigns separate numeric scores on multiple dimensions simultaneously.

**When to use multi-metric scoring:**
- Outputs can be partially correct (accuracy) but poorly structured (clarity)
- Multiple stakeholder priorities must be balanced (helpfulness vs. conciseness)
- You need a weighted composite score to rank competing prompt variants

### Example: Customer support response quality

We evaluate responses on three dimensions (0–5 each):

| Metric | Meaning |
|--------|---------|
| `accuracy` | Factual correctness — does the answer correctly address the question? |
| `helpfulness` | Actionability — does the user know what to do next? |
| `tone` | Appropriate for customer-facing context (professional, empathetic) |

**Weighted composite**: `0.5 × accuracy + 0.3 × helpfulness + 0.2 × tone`

> **Key insight**: a weighted composite reduces multi-dimensional performance to a single comparable number, enabling A/B ranking. But always inspect per-dimension scores — a high composite can mask a critical failure on one axis.

In [ ]:
// Multi-metric scoring: grade on accuracy, helpfulness, and tone (each 0–5)
type MetricScore = { accuracy: number; helpfulness: number; tone: number };

function buildMultiMetricGraderPrompt(
  question: string,
  answer: string
): Anthropic.MessageParam[] {
  return [{
    role: 'user',
    content:
      `You are an expert evaluator assessing a customer support AI response.\n\n` +
      `<question>${question}</question>\n\n` +
      `<answer>${answer}</answer>\n\n` +
      `Score the answer on three dimensions (each 0–5):\n` +
      `- accuracy: Is the answer factually correct and complete? (0=wrong, 5=fully correct)\n` +
      `- helpfulness: Does the user know what to do after reading this? (0=useless, 5=very actionable)\n` +
      `- tone: Is the language professional and empathetic? (0=inappropriate, 5=excellent)\n\n` +
      `First reason inside <thinking></thinking> tags.\n` +
      `Then output exactly this JSON inside <scores></scores> tags:\n` +
      `{"accuracy": X, "helpfulness": X, "tone": X}`,
  }];
}

async function gradeMultiMetric(
  question: string,
  answer: string
): Promise<MetricScore> {
  const completion = await getCompletion(buildMultiMetricGraderPrompt(question, answer), 512);
  const match = completion.match(/<scores>([\s\S]*?)<\/scores>/);
  if (!match) throw new Error('Grader did not return <scores> tags');
  return JSON.parse(match[1].trim()) as MetricScore;
}

function compositeScore(s: MetricScore): number {
  // Weighted: accuracy matters most, tone least
  return 0.5 * s.accuracy + 0.3 * s.helpfulness + 0.2 * s.tone;
}

// Eval dataset: customer support questions
const supportEval: Array<{ question: string }> = [
  { question: "My order hasn't arrived after 10 days. What should I do?" },
  { question: 'Can I return a product I bought 45 days ago?' },
  { question: 'My account is locked. How do I reset my password?' },
];

const supportOutputs = await Promise.all(
  supportEval.map(q => getCompletion([{ role: 'user', content: q.question }]))
);

const multiMetricGrades = await Promise.all(
  supportOutputs.map((out, i) => gradeMultiMetric(supportEval[i].question, out))
);

console.log('=== Multi-metric Scoring Results ===\n');
multiMetricGrades.forEach((scores, i) => {
  const composite = compositeScore(scores);
  console.log(`Q${i + 1}: ${supportEval[i].question}`);
  console.log(
    `  accuracy=${scores.accuracy}/5  helpfulness=${scores.helpfulness}/5  tone=${scores.tone}/5`
  );
  console.log(`  Weighted composite: ${composite.toFixed(2)} / 5.00\n`);
});

const avgComposite =
  multiMetricGrades.reduce((sum, s) => sum + compositeScore(s), 0) / multiMetricGrades.length;
console.log(`Overall composite score: ${avgComposite.toFixed(2)} / 5.00`);

## Section 6: Eval Dataset Design Strategy

The quality of your eval dataset determines how trustworthy your scores are. Two dimensions matter: **size** (number of examples) and **quality** (representativeness and rubric clarity).

### Size vs. quality trade-off

| Dataset Size | Reliability | Practical Guidance |
|-------------|-------------|-------------------|
| < 50 examples | Low — high variance | Use only for rapid prototyping |
| 50–100 examples | Moderate | Acceptable for prompt iteration cycles |
| 200–500 examples | High | Recommended for production CI gates |
| 500+ examples | Very high | Needed for fine-tuning decisions |

**Rule of thumb**: 200 diverse examples beat 20 carefully crafted ones — statistical coverage trumps individual perfection.

### Generating synthetic eval data with Claude

When hand-labeling is expensive, use Claude to generate a diverse dataset:

1. Write a few seed examples by hand (5–10)
2. Describe your task to Claude with those seeds
3. Ask Claude to produce many variants: easy, medium, hard, edge cases
4. Filter and deduplicate the output before use

> **Caution**: synthetic data can encode Claude's own blind spots. Always validate a sample (10–20%) with human labels before using the dataset as a CI gate.

In [ ]:
// Generate a synthetic eval dataset using Claude
async function generateSyntheticEval(
  taskDescription: string,
  seedExamples: string,
  count: number
): Promise<Array<{ input: string; goldenAnswer: string }>> {
  const prompt: Anthropic.MessageParam[] = [{
    role: 'user',
    content:
      `You are an eval dataset designer. Generate ${count} diverse evaluation questions for this task:\n\n` +
      `<task>${taskDescription}</task>\n\n` +
      `<seeds>\n${seedExamples}\n</seeds>\n\n` +
      `Requirements:\n` +
      `- Mix of easy, medium, and hard questions\n` +
      `- Include at least 2 edge cases\n` +
      `- No duplicates or near-duplicates\n\n` +
      `Output ONLY a JSON array with no explanation:\n` +
      `[{"input": "...", "goldenAnswer": "..."}, ...]`,
  }];

  const completion = await getCompletion(prompt, 2048);
  const match = completion.match(/\[[\s\S]*\]/);
  if (!match) throw new Error('Could not extract JSON array from response');
  return JSON.parse(match[0]) as Array<{ input: string; goldenAnswer: string }>;
}

// Example: generate geography trivia questions
const taskDesc = 'Answer geography questions. Return only the city or country name, nothing else.';
const seeds = [
  'Q: What is the capital of France? A: Paris',
  'Q: What is the capital of Japan? A: Tokyo',
  'Q: What country is Sydney in? A: Australia',
].join('\n');

const syntheticDataset = await generateSyntheticEval(taskDesc, seeds, 8);

console.log(`Generated ${syntheticDataset.length} synthetic eval examples:\n`);
syntheticDataset.forEach((ex, i) => {
  console.log(`  ${i + 1}. Input:  "${ex.input}"`);
  console.log(`     Golden: "${ex.goldenAnswer}"`);
});

// Size reliability helper
function assessDatasetSize(n: number): string {
  if (n < 50)  return `WARNING: ${n} examples — low reliability, high variance`;
  if (n < 200) return `OK:      ${n} examples — acceptable for iteration`;
  if (n < 500) return `GOOD:    ${n} examples — reliable, suitable for CI gate`;
  return       `BEST:    ${n} examples — very high reliability`;
}

console.log('\nDataset size reliability guide:');
[8, 50, 200, 500].forEach(n => console.log(' ', assessDatasetSize(n)));

## Section 7: A/B Prompt Testing Workflow

A/B prompt testing runs the **same eval dataset** through two prompt variants and compares their scores. It is the primary tool for data-driven prompt iteration.

### Workflow

```
Eval dataset  →  [Prompt A]  →  Outputs A  →  Grader  →  Score A
              →  [Prompt B]  →  Outputs B  →  Grader  →  Score B
                                                       →  Compare Δ
```

### Interpreting results

| Score Δ | Small eval (<100 questions) | Large eval (>200 questions) |
|---------|-----------------------------|-----------------------------|
| < 5%    | Likely noise — re-run before deciding | Borderline — inspect per-question variance |
| 5–15%   | Suggestive — worth a follow-up run | Meaningful — lean toward the winner |
| > 15%   | Strong signal even at small scale | Decisive — ship the winner |

> **Important**: always grade both variants with the **same grader** in the same run. Mixing grader versions introduces a confound that makes the delta meaningless.

In [ ]:
// A/B prompt testing: compare two system prompts on the same eval dataset
const PROMPT_A =
  `You are a helpful assistant. Answer questions accurately and concisely.`;

const PROMPT_B =
  `You are an expert assistant. Before answering, briefly consider edge cases, ` +
  `then give a precise and complete answer.`;

async function runWithSystemPrompt(
  systemPrompt: string,
  question: string
): Promise<string> {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 512,
    system: systemPrompt,
    messages: [{ role: 'user', content: question }],
  });
  return (response.content[0] as Anthropic.TextBlock).text;
}

// Reuse the open-ended eval dataset from Section 2
const abEval = openEval;

console.log('Running A/B test — generating outputs for both prompts...');

const [outputsA, outputsB] = await Promise.all([
  Promise.all(abEval.map(q => runWithSystemPrompt(PROMPT_A, q.question))),
  Promise.all(abEval.map(q => runWithSystemPrompt(PROMPT_B, q.question))),
]);

const [gradesA, gradesB] = await Promise.all([
  Promise.all(outputsA.map((out, i) => gradeWithModel(out, abEval[i].goldenAnswer))),
  Promise.all(outputsB.map((out, i) => gradeWithModel(out, abEval[i].goldenAnswer))),
]);

const scoreA = (gradesA.filter(g => g === 'correct').length / gradesA.length) * 100;
const scoreB = (gradesB.filter(g => g === 'correct').length / gradesB.length) * 100;
const delta = scoreB - scoreA;

console.log('\n=== A/B Test Results ===\n');
console.log(`Prompt A: ${scoreA.toFixed(1)}%`);
console.log(`Prompt B: ${scoreB.toFixed(1)}%`);
console.log(`Delta (B - A): ${delta > 0 ? '+' : ''}${delta.toFixed(1)}%`);
console.log(`Winner: Prompt ${scoreB > scoreA ? 'B' : scoreB < scoreA ? 'A' : 'Tie'}\n`);

console.log('Per-question breakdown:');
abEval.forEach((q, i) => {
  const a = gradesA[i] === 'correct' ? 'pass' : 'fail';
  const b = gradesB[i] === 'correct' ? 'pass' : 'fail';
  console.log(`  Q${i + 1}: A=${a}  B=${b}  | ${q.question.slice(0, 60)}...`);
});

console.log('\nNote: with only', abEval.length, 'questions, treat this result as directional.');
console.log('Use >= 100 questions for reliable A/B conclusions.');

## Summary: Choosing a Grading Method

| Scenario | Recommended Method |
|----------|-------------------|
| Output is a number, label, or parseable JSON | Code-based (exact match / regex) |
| Short classification — yes/no, A/B/C/D | Code-based |
| Open-ended text; rubric is objective | Model-based (Claude as judge) |
| Open-ended text; rubric is subjective | Model-based + human calibration sample |
| Safety-critical or highly nuanced | Human |
| Calibrating a new model grader | Human sample → then model-based |
| Multiple quality dimensions matter | Multi-metric scoring (Section 5) |
| Comparing two prompt variants | A/B prompt testing (Section 7) |

### Cost rule of thumb (100-question eval run)

| Method | Approx. Cost |
|--------|--------------|
| Code-based | ~$0 extra |
| Model-based with Haiku | ~$0.01–$0.05 |
| Model-based with Sonnet | ~$0.10–$0.50 |
| Human (contractor rates) | ~$50–$200+ |

### Dataset size guidance

| Size | Reliability |
|------|-------------|
| < 50 | Low — high variance |
| 50–100 | Acceptable for prompt iteration |
| 200–500 | Reliable — suitable for CI gate |
| 500+ | High-confidence decisions |

Building systematic evals **early** saves enormous debugging time later — every prompt change becomes a measurable experiment rather than a guess.